# singular-matrix-mask-trick — faded example 2: Clone and overwrite singular slices with the identity

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `singular-matrix-mask-trick`. The last cell reports your progress on the `Numpy: Singular matrix mask trick` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Singular matrix mask trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`singular-matrix-mask-trick`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "singular-matrix-mask-trick"
DD_SUBTOPIC = "Numpy: Singular matrix mask trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Once you have the singularity mask, you need to (1) clone the input batch so you don't mutate the caller's tensor, then (2) use boolean indexing to overwrite each singular slice with the identity matrix. The identity matrix is invertible (determinant = 1), so the patched batch can be passed to `linalg.solve` without raising an error.

## Faded exercise 2

Given A (K, n, n) and is_singular (K,) boolean, produce A_safe: a clone of A where every singular slice is replaced by the n×n identity matrix.

1. Clone A.
2. Overwrite singular slices.

The blank step is overwriting the singular slices of A_safe with the identity.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

n = 2
A = t.zeros(4, n, n)
A[0] = t.tensor([[2.0, 1.0], [0.0, 3.0]])
A[1] = t.tensor([[1.0, 2.0], [2.0, 4.0]])  # singular
A[2] = t.tensor([[5.0, 2.0], [1.0, 3.0]])
A[3] = t.tensor([[3.0, 6.0], [0.5, 1.0]])  # singular
is_singular = t.tensor([False, True, False, True])

A_safe = A.clone()
raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above

print('A_safe[1]:', A_safe[1].tolist())   # should be [[1,0],[0,1]]
print('A_safe[3]:', A_safe[3].tolist())   # should be [[1,0],[0,1]]
print('A_safe[0]:', A_safe[0].tolist())   # should be unchanged [[2,1],[0,3]]


def _test():
    import torch as t

    n = 2
    A_orig = t.zeros(4, n, n)
    A_orig[0] = t.tensor([[2.0, 1.0], [0.0, 3.0]])
    A_orig[1] = t.tensor([[1.0, 2.0], [2.0, 4.0]])
    A_orig[2] = t.tensor([[5.0, 2.0], [1.0, 3.0]])
    A_orig[3] = t.tensor([[3.0, 6.0], [0.5, 1.0]])
    is_singular = t.tensor([False, True, False, True])

    eye = t.eye(n, dtype=A_orig.dtype)
    assert t.allclose(A_safe[1], eye), f'A_safe[1] should be identity, got {A_safe[1]}'
    assert t.allclose(A_safe[3], eye), f'A_safe[3] should be identity, got {A_safe[3]}'
    assert t.allclose(A_safe[0], A_orig[0]), 'non-singular slices should be unchanged'
    assert t.allclose(A_safe[2], A_orig[2]), 'non-singular slices should be unchanged'
    # original A should not have been mutated
    assert t.allclose(A[1], t.tensor([[1.0, 2.0], [2.0, 4.0]])), 'original A was mutated'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

n = 2
A = t.zeros(4, n, n)
A[0] = t.tensor([[2.0, 1.0], [0.0, 3.0]])
A[1] = t.tensor([[1.0, 2.0], [2.0, 4.0]])
A[2] = t.tensor([[5.0, 2.0], [1.0, 3.0]])
A[3] = t.tensor([[3.0, 6.0], [0.5, 1.0]])
is_singular = t.tensor([False, True, False, True])

A_safe = A.clone()
A_safe[is_singular] = t.eye(n, dtype=A.dtype)

print('A_safe[1]:', A_safe[1].tolist())
print('A_safe[3]:', A_safe[3].tolist())
print('A_safe[0]:', A_safe[0].tolist())
```
</details>